# Upgrade a Lista de Entregas

In [319]:
from pathlib import Path
import pandas as pd
import json
import datetime as dt
import re
from difflib import get_close_matches

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
print('Importadas ok')

Importadas ok


# Input Collect Datos

## 1) Lista de Entregas
Fenix

In [320]:
carpeta = "../data/lista_entregas/"
file = carpeta + "CRONOGRAMA_ENTREGAS_FENIX_HARUMI.xlsx"

In [321]:
df_entregas_raw = pd.ExcelFile(file)
print(f'Pestañas: {df_entregas_raw.sheet_names}')


df_entregas = pd.read_excel(file, sheet_name="CRONO-ORD-FECHA", skiprows=2, dtype={'DNI': str})

print(f'Total Entregas Pestaña 1: {df_entregas.shape[0]} clientes')
df_entregas.head()

Pestañas: ['CRONO-ORD-FECHA', 'CRONO-ORD-PISO', 'EN PROCESO', 'LIBRES']
Total Entregas Pestaña 1: 39 clientes


,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN
2,3.0,DEPARTAMENTO,1503,DEL VALLE VARILLAS JOSUE HAMILTON,71300365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Ago 2025,12 Sep 2025,20 Jul 2026,2:00 pm,NaN,NaN
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN
4,NaN,ESTACIONAMIENTO,37,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Sep 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN


In [322]:
df_entregas

,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN
2,3.0,DEPARTAMENTO,1503,DEL VALLE VARILLAS JOSUE HAMILTON,71300365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Ago 2025,12 Sep 2025,20 Jul 2026,2:00 pm,NaN,NaN
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN
4,NaN,ESTACIONAMIENTO,37,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Sep 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN
5,5.0,DEPARTAMENTO,804,ROMO AGUIRRE LUIGUI ROLANDO,74827647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20 Dic 2024,02 Oct 2025,22 Jul 2026,9:00 am,NaN,NaN
6,6.0,DEPARTAMENTO,704,LLANQUE FRAQUITA MANINO HELAMAN,40654929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Jul 2025,14 Oct 2025,22 Jul 2026,11:00 am,NaN,NaN
7,7.0,DEPARTAMENTO,1705,BULEJE CALDERON WILDER WILFREDO,71477258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,30 Oct 2025,22 Jul 2026,2:00 pm,NaN,NaN
8,8.0,DEPARTAMENTO,303,ALDERETE SOLIS SADITH PATRICIA,46137521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,31 Oct 2025,22 Jul 2026,4:00 pm,NaN,NaN
9,9.0,DEPARTAMENTO,802,BELLIDO PANTOJA ROSA MARIA,73097248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21 Ago 2025,24 Dic 2025,24 Jul 2026,9:00 am,NaN,NaN


In [323]:
df_procesos = pd.read_excel(file, sheet_name="EN PROCESO", skiprows=2)

print(f'Total Procesos Pestaña 2: {df_procesos.shape[0]} clientes')
df_procesos.head()

Total Procesos Pestaña 2: 66 clientes


,#,Tipo,N° Unidad,Comprador,% Avance,F. Separación
0,1,ESTACIONAMIENTO,7,BELLIDO PANTOJA ROSA MARIA,0.770,24 Jun 2025
1,2,DEPARTAMENTO,402,BLAZ MARIÑO FRANK,0.100,06 Jun 2026
2,3,DEPARTAMENTO,1201,CACERES VALVERDE HENRY ENRIQUE,0.003,16 Jun 2026
3,4,DEPARTAMENTO,903,CAMAHUALI HUAMAN NANCY LIZ,0.249,06 Abr 2025
4,5,ESTACIONAMIENTO,32,CAMAHUALI HUAMAN NANCY LIZ,0.000,NaN


In [324]:
import numpy as np

# 1) Asegurar nombres limpios
df_procesos.columns = df_procesos.columns.astype(str).str.strip()

# 2) Limpiar N° Unidad para evitar valores tipo 1201.0
df_procesos["unidad_key"] = (
    df_procesos["N° Unidad"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# 3) Normalizar Tipo
tipo_key = (
    df_procesos["Tipo"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 4) Crear codigo_unidad
df_procesos["codigo_unidad"] = np.where(
    tipo_key.eq("ESTACIONAMIENTO"),
    "FX-E" + df_procesos["unidad_key"],
    "FX-" + df_procesos["unidad_key"]
)

df_procesos[["Tipo", "N° Unidad", "codigo_unidad"]].head()

,Tipo,N° Unidad,codigo_unidad
0,ESTACIONAMIENTO,7,FX-E7
1,DEPARTAMENTO,402,FX-402
2,DEPARTAMENTO,1201,FX-1201
3,DEPARTAMENTO,903,FX-903
4,ESTACIONAMIENTO,32,FX-E32


## 2) Clientes Sperant

In [325]:
carpeta = "../data/raw/"
file = carpeta + "clientes.parquet"

In [326]:
df_raw = pd.read_parquet(file)
df_raw.head()

,fecha_creacion,nombres,apellidos,tipo_documento,documento,genero,estado_civil,email,telefono,celulares,agrupacion_medio_captacion,medio_captacion,canal_entrada,nivel_interes,fecha_nacimiento,nacionalidad,pais,departamento,provincia,distrito,direccion,apto,observacion,ocupacion,documento_conyuge,usuario_creador,username,estado,ultimo_proyecto,total_unidades_asignadas,ultimo_vendedor,total_interacciones,fecha_ultima_interaccion,proyectos_relacionados,id,codigo_externo_cliente,rango_edad,origen,ultimo_tipo_interaccion,fecha_actualizacion,autorizacion_uso_datos,autorizacion_publicidad,geo_latitud,geo_longitud,geolocalizacion,cliente_riesgo,agrupacion_canal_entrada,tipo_persona,denominacion,tipo_financiamiento,desistido,razon_desistimiento,hora_creacion,fecha_primera_interaccion_manual,prioridad,uuid
0,2021-10-23,Eliana,.,DNI,42474219,Femenino,None,maeligolu@hotmail.com,None,+51966848301,None,verdancy,landing,intermedio,None,None,PE,LIM,None,None,None,Apto,None,None,None,None,None,Interesado,Torre Nápoles,0,Antonio Martinez,9,2026-05-07,"NP,GY,001",43002,None,None,api,whatsapp,2026-05-07,True,True,None,None,None,no,None,Persona Natural,None,None,no,None,12:49:07,2021-10-23 17:52:01.696182,None,161eb462-f2d0c1-e315
1,2025-03-15,JORGE,PAZ PRINCIPE,DNI,70474161,Masculino,soltero,PAZ_PRINCIPE_JORGE@HOTMAIL.COM,,+51987599704,None,feria inmobiliaria,whatsapp,en espera,None,peruvian,PE,LIM,Lima,None,,Apto,,,None,Nelly Román Villavicencio,nroman,Interesado,Sialia,1,Nelly Román Villavicencio,36,2026-05-07,SL,109614,None,None,manual,whatsapp,2026-05-07,True,True,None,None,None,no,None,Persona Natural,Sr,hipotecario,no,None,19:00:38,2025-03-16 00:03:01.522276,None,6a646088-daa3fe-0d9a
2,2022-01-24,Milagros,Dominguez Medianero,DNI,71858180,Femenino,soltero,mili03r@gmail.com,,+51943260635,None,facebook,contacto web,por contactar,None,peruvian,PE,LIM,Lima,None,,Apto,observation: El tamaño del departamento,,None,None,None,Interesado,Torre Nápoles,3,Ana Lucia DelSolar,14,2026-05-07,"NP,FX,GY,URT",48395,None,None,fblead,whatsapp,2026-05-07,True,True,None,None,None,no,None,Persona Natural,,hipotecario,no,None,14:36:21,2022-01-24 22:41:34.663772,None,af9437df-12b8af-186e
3,2022-01-11,Lourdes,Escudero Reynoso,DNI,08490083,Femenino,soltero,escuderoreynoso@gmail.com,,+51979339267,None,feria inmobiliaria,visita presencial,por contactar,None,peruvian,PE,LIM,Lima,None,,Apto,,,None,Ornella Muñoz Alfaro,omunoz,Interesado,Torre Nápoles,4,Ana Lucia DelSolar,6,2026-05-07,"SL,NP,GY,001,MA",47691,None,None,manual,whatsapp,2026-05-07,True,True,None,None,None,no,None,Persona Natural,Sra,crédito directo,no,None,18:20:36,2022-01-11 23:44:54.128225,None,9e026a4d-86f429-c1e0
4,2020-01-20,JUAN,MENDEZ,DNI,auto-688312,Masculino,soltero,Jrpatriniz@gmail.com,+51958496148,+51958496148,None,facebook,por email,por contactar,None,peruvian,PE,LIM,Lima,None,,Apto,¿qué_es_lo_que_más_importante_para_elegir_un_departamento? : Financiamiento <br/>,,None,None,None,Interesado,Sialia,0,Nelly Román Villavicencio,63,2026-05-07,"MA,SL,CUBA",4166,None,None,fblead,whatsapp,2026-05-07,True,True,None,None,None,no,None,Persona Natural,,None,no,None,18:29:49,2023-07-27 22:47:46.777730,None,58a2d751-748722-2670


### Propietarios

In [327]:
df_propietarios = df_raw[df_raw['estado']=="Propietario"]
print(f'Total Propietarios: {df_propietarios.shape[0]} clientes')
col_contacto = ['documento', 'celulares', 'email', 'telefono', 'nombres', 'apellidos']
df_propietarios = df_propietarios[col_contacto]
df_propietarios
df_propietarios.sample(2)


Total Propietarios: 1279 clientes


,documento,celulares,email,telefono,nombres,apellidos
95190,31039192,+51906406030,consueloperez03031973@gmail.com,,JESSICA CONSUELO,PEREZ MORALES
85509,19994740,+51959089902,JCGA377@HOTMAIL.COM,,AMPARO,ALIAGA MIRANDA DE GARCIA


### Calidad de celular

In [328]:
def limpiar_celular(valor):
    if pd.isna(valor):
        return pd.NA
    
    s = str(valor).strip()
    
    # Dejar solo números
    s = re.sub(r"\D", "", s)
    
    # Quitar prefijo Perú si viene como 51XXXXXXXXX
    if s.startswith("51") and len(s) == 11:
        s = s[2:]
    
    # Quitar prefijo Perú si viene como 0051XXXXXXXXX
    if s.startswith("0051") and len(s) == 13:
        s = s[4:]
    
    # Si quedó vacío
    if s == "":
        return pd.NA
    
    return s

In [329]:
df_propietarios["celular_clean"] = df_propietarios["celulares"].apply(limpiar_celular)

df_propietarios["celular_len"] = df_propietarios["celular_clean"].astype("string").str.len()

df_propietarios["celular_ok"] = (
    (df_propietarios["celular_len"] == 9) &
    (df_propietarios["celular_clean"].astype("string").str.startswith("9"))
)

In [330]:
df_propietarios["celular_ok"].value_counts(dropna=False)

celular_ok
True     1011
<NA>      145
False     123
Name: count, dtype: Int64

In [331]:
df_propietarios.loc[
    ~df_propietarios["celular_ok"].fillna(False),
    ["celulares", "celular_clean", "celular_len"]
].head(30)

,celulares,celular_clean,celular_len
560,+420773621533,420773621533,12
1733,None,<NA>,<NA>
2008,,<NA>,<NA>
2805,+517878787878,517878787878,12
4055,+511254s4s4s4,511254444,9
5702,None,<NA>,<NA>
5704,None,<NA>,<NA>
5705,None,<NA>,<NA>
6209,+51124569878,124569878,9
6306,None,<NA>,<NA>


### Calidad de Email

In [332]:
DOMINIOS_CORRECTOS = [
    "gmail.com",
    "hotmail.com",
    "outlook.com",
    "yahoo.com",
    "icloud.com",
    "live.com",
    "msn.com",
    "contraloria.gob.pe",
]

MAPA_TYPOS_DOMINIO = {
    "gmai.com": "gmail.com",
    "gmal.com": "gmail.com",
    "gmial.com": "gmail.com",
    "gmail.con": "gmail.com",
    "gmail.co": "gmail.com",
    "gmail.cm": "gmail.com",
    "gmail.comm": "gmail.com",

    "hotmai.com": "hotmail.com",
    "hotmial.com": "hotmail.com",
    "homtail.com": "hotmail.com",
    "hotmail.con": "hotmail.com",
    "hotmail.co": "hotmail.com",
    "hotmail.cm": "hotmail.com",
    "hotmail.comm": "hotmail.com",

    "outlok.com": "outlook.com",
    "outloo.com": "outlook.com",
    "outlook.con": "outlook.com",

    "yaho.com": "yahoo.com",
    "yahoo.con": "yahoo.com",
}

In [333]:
def limpiar_email(valor):
    if pd.isna(valor):
        return pd.NA
    
    s = str(valor).strip().lower()
    
    # Quitar espacios internos
    s = re.sub(r"\s+", "", s)
    
    # Correcciones frecuentes
    s = s.replace(",", ".")
    s = s.replace("..", ".")
    
    # Si no tiene arroba, no se puede corregir con seguridad
    if "@" not in s:
        return s
    
    partes = s.split("@")
    
    # Si tiene más de un @, dejamos el primero como usuario y el último como dominio
    usuario = partes[0]
    dominio = partes[-1]
    
    # Quitar puntos raros al inicio/final
    usuario = usuario.strip(".")
    dominio = dominio.strip(".")
    
    # Corrección directa por mapa
    if dominio in MAPA_TYPOS_DOMINIO:
        dominio = MAPA_TYPOS_DOMINIO[dominio]
    else:
        # Corrección aproximada solo para dominios comunes
        match = get_close_matches(dominio, DOMINIOS_CORRECTOS, n=1, cutoff=0.88)
        if match:
            dominio = match[0]
    
    return f"{usuario}@{dominio}"

In [334]:
df_propietarios["email_clean"] = df_propietarios["email"].apply(limpiar_email)

In [335]:
patron_email = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

df_propietarios["email_ok"] = (
    df_propietarios["email_clean"]
    .astype("string")
    .str.match(patron_email, na=False)
)

In [336]:
df_propietarios.loc[
    df_propietarios["email"].astype("string").str.lower() != df_propietarios["email_clean"].astype("string"),
    ["email", "email_clean", "email_ok"]
].head(30)

,email,email_clean,email_ok
6209,1226652@hotmail.colm,1226652@hotmail.com,True
9223,andresrivasplatacastro@gmal.com,andresrivasplatacastro@gmail.com,True
9339,anamelbaquisperegular@gmail.om,anamelbaquisperegular@gmail.com,True
21940,arovi50@hotmil.com,arovi50@hotmail.com,True
22353,notiene@gmai1l.com,notiene@gmail.com,True
25027,christian.pablo24@gmail.con,christian.pablo24@gmail.com,True
25584,yesseniasotaf@hotmai.com,yesseniasotaf@hotmail.com,True
28122,jenny@gmaiil.com,jenny@gmail.com,True
82681,emiliaicochea@gmauil.com,emiliaicochea@gmail.com,True
83258,carmen7_@outlook.com.pe,carmen7_@outlook.com,True


In [337]:
df_propietarios.loc[
    df_propietarios["email"].astype("string").str.lower() != df_propietarios["email_clean"].astype("string"),
    ["email", "email_clean", "email_ok"]
].head(30)

,email,email_clean,email_ok
6209,1226652@hotmail.colm,1226652@hotmail.com,True
9223,andresrivasplatacastro@gmal.com,andresrivasplatacastro@gmail.com,True
9339,anamelbaquisperegular@gmail.om,anamelbaquisperegular@gmail.com,True
21940,arovi50@hotmil.com,arovi50@hotmail.com,True
22353,notiene@gmai1l.com,notiene@gmail.com,True
25027,christian.pablo24@gmail.con,christian.pablo24@gmail.com,True
25584,yesseniasotaf@hotmai.com,yesseniasotaf@hotmail.com,True
28122,jenny@gmaiil.com,jenny@gmail.com,True
82681,emiliaicochea@gmauil.com,emiliaicochea@gmail.com,True
83258,carmen7_@outlook.com.pe,carmen7_@outlook.com,True


## 3) Procesos de Separación

In [338]:
carpeta = "../data/raw/"
file = carpeta + "procesos.parquet"

In [339]:
df_raw = pd.read_parquet(file)
df_raw.head()

,codigo_proyecto,nombre_proyecto,tipo_unidad_principal,codigo_unidad,total_unidades,codigo_unidades_asignadas,nombres_cliente,apellidos_cliente,documento_cliente,origen_proforma,fecha_proforma,codigo_proforma,numero_contrato,fecha_contrato,modalidad_contrato,moneda,tipo_cambio,tipo_financiamiento,banco,situacion_legal,documento_representante,nombres_usuario,username,precio_base_proforma,descuento_venta,precio_venta,aprobador_descuento,nombre,premios,fecha_inicio,fecha_fin,fecha_expiracion,fecha_impresion_contrato,nombre_flujo,estado,completado,total_pagado,total_pendiente,fecha_analisis,fecha_nif,estado_nif,utm_source,utm_medium,utm_campaign,utm_term,utm_content,documento_conyuge,documento_copropietarios,flujo_anulacion,fecha_anulacion,codigo_externo_venta,id,tipo,fecha_actualizacion,codigo_externo_minuta,flujo_error,momento_caida,tipo_cronograma,estado_contrato,devolucion,fecha_devolucion,excedente,observacion_devolucion,motivo_caida,nombres_usuario_aprobador,username_aprobador,cliente_id,usuario_creador,username_creador,usuario_separacion,codigo_externo_entrega,fecha_minuta,proforma_id,penalidad,proceso_anulacion,codigo_externo_anulacion_venta,terminado,paso_actual,estado_personalizado
0,EEUU,Edificio Urbanzen,departamento flat,EEUU-301,1,None,Edith Gloria,Valenzuela,25550885,manual,2023-09-30,2023-02274,None,None,Compromiso,PEN,1.000000,hipotecario,None,Conyugal,00085850,Enzo Villanueva,evillanueva,570476.00,0.00,570476.00,None,Entrega,None,2026-01-26,2026-01-26,None,None,Proceso de entrega,Activo,Completado,0.00,2500.00,2026-01-26,None,None,None,None,None,None,None,00085850,None,None,None,None,20,ENTREGA,2026-01-30,None,False,None,Hipotecario,None,0.00,None,0.0,None,None,None,None,82237,JOSUE QUIROGA NAVARRO,jquiroga,None,None,None,16356,0.00,al iniciar,None,None,Encuestas de propietarios,None
1,TZ,Tizón y Bueno,departamento flat,TZ-603,1,None,Renzo Paolo,Huarcaya Vasquez,72324651,manual,2025-08-11,2025-04702,None,None,Compromiso,PEN,1.000000,hipotecario,Banco de Crédito del Perú (BCP),Propietario Solo,22097481,Luis Peñaloza,lpenaloza,484267.78,48426.78,435841.00,ddinatale,Entrega,None,2026-01-05,None,None,None,Proceso de entrega,Activo,No Completado,0.00,0.00,None,None,None,None,None,None,None,None,None,None,None,None,None,16,ENTREGA,2026-01-05,None,False,None,None,None,0.00,None,0.0,None,None,None,None,43358,JOSUE QUIROGA NAVARRO,jquiroga,None,None,None,25011,0.00,al iniciar,None,None,Encuestas de propietarios,None
2,TZ,Tizón y Bueno,departamento flat,TZ-1504,1,None,Ana Rosario,Zevallos Machuca de Rivero,07733980,manual,2025-04-10,2025-02021,None,None,Compromiso,PEN,1.000000,hipotecario,None,Propietario Solo,583650117,Joceline Arizabal Corimanya,jarizabal,455333.33,46333.33,409000.00,ddinatale,Entrega,None,2026-01-09,None,None,None,Proceso de entrega,Activo,No Completado,0.00,0.00,None,None,None,None,None,None,None,None,583650117,None,None,None,None,17,ENTREGA,2026-01-09,None,False,None,None,None,0.00,None,0.0,None,None,None,None,111636,JOSUE QUIROGA NAVARRO,jquiroga,None,None,None,22329,0.00,al iniciar,None,None,Encuestas de propietarios,None
3,EEUU,Edificio Urbanzen,departamento flat,EEUU-701,3,"EEUU-D5,EEUU-E34",ZAIDA LIZ,GUERRA ZERPA,44612849,manual,2024-09-07,2024-02526,None,None,Compromiso,PEN,1.000000,hipotecario,Banco de Crédito del Perú (BCP),Propietario Solo,None,Yomira Pacheco,ypacheco,638000.00,0.00,638000.00,None,Entrega,None,2026-01-30,2026-01-30,None,None,Proceso de entrega,Activo,Completado,0.00,0.00,2026-01-30,None,None,None,None,None,None,None,None,None,None,None,None,43,ENTREGA,2026-02-12,None,False,None,None,None,0.00,None,0.0,None,None,None,None,89900,Diego Di Natale,ddinatale,None,None,None,18936,0.00,al iniciar,None,None,Proceso Terminado,None
4,EEUU,Edificio Urbanzen,departamento flat,EEUU-1503,1,None,Nahomy Elizabeth,Casildo Arteaga,74390653,manual,2025-07-12,2025-04107,None,None,Compromiso,PEN,1.000000,hipotecario,None,Propietario Solo,None,Yomira Pacheco,ypacheco,362520.00,36520.00,326000.00,ev

### Separaciones

In [340]:
df_raw['nombre_proyecto'].value_counts()

nombre_proyecto
Edificio Cuba Connect           582
Tizón y Bueno                   527
Edificio Santa Cruz Infinite    400
Edificio Saenz                  376
Torre Nápoles                   346
Edificio Mariategui             336
Edificio Urbanzen               311
Modena                          304
Alicanto                        293
Sialia                          284
Fenix                           275
Matera                          222
Capadocia                       204
Los Jardines de Urteaga         185
Edificio Valdizan                91
Tradiciones Prime                12
Edificio Unique                   4
Name: count, dtype: int64

In [341]:
df_separaciones = df_raw[df_raw['nombre_proyecto']=="Fenix"]
df_separaciones = df_separaciones[df_separaciones['estado']=="Activo"]
df_separaciones = df_separaciones[df_separaciones['nombre']=="Separacion"]
print(f'Total Separaciones Activas: {df_separaciones.shape[0]} clientes')
col_contacto = ['nombre_proyecto','tipo_unidad_principal', 'codigo_unidad' , 'documento_cliente', 'nombres_usuario']
df_separaciones = df_separaciones[col_contacto]
#df_separaciones
df_separaciones.sample(80)


Total Separaciones Activas: 84 clientes


,nombre_proyecto,tipo_unidad_principal,codigo_unidad,documento_cliente,nombres_usuario
1817,Fenix,departamento flat,FX-405,41441720,Karlo Vargas
2817,Fenix,departamento flat,FX-1005,72945691,Ana Lucia DelSolar
4294,Fenix,departamento flat,FX-602,46816338,Yomira Pacheco
595,Fenix,departamento flat,FX-1102,45942433,Luis Peñaloza
491,Fenix,departamento flat,FX-1302,31182859,Joceline Arizabal Corimanya
...,...,...,...,...,...
2418,Fenix,departamento flat,FX-1301,25003673,Percy Soto
2316,Fenix,departamento duplex,FX-1702,47160251,Ana Lucia DelSolar
2258,Fenix,departamento flat,FX-604,10350646,Ivan La Riva
2831,Fenix,estacionamiento simple,FX-E5,06766241,Joceline Arizabal Corimanya


In [342]:
df_separaciones['tipo_unidad_principal'].value_counts()

tipo_unidad_principal
departamento flat               69
estacionamiento con depósito     8
estacionamiento simple           6
departamento duplex              1
Name: count, dtype: int64

In [343]:
df_separaciones.info()
#df_separaciones['nombre'].value_counts()

<class 'pandas.core.frame.DataFrame'>
Index: 84 entries, 183 to 4725
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   nombre_proyecto        84 non-null     object
 1   tipo_unidad_principal  84 non-null     object
 2   codigo_unidad          84 non-null     object
 3   documento_cliente      84 non-null     object
 4   nombres_usuario        84 non-null     object
dtypes: object(5)
memory usage: 3.9+ KB


# Merge con Sperant

## 1) Main

In [344]:
print(f'Total Entregas Pestaña 1: {df_entregas.shape[0]} clientes')
df_entregas.sample(2)

Total Entregas Pestaña 1: 39 clientes


,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.
22,20.0,DEPARTAMENTO,205,TORRES SOTELO MILAGROS MATILDE,45063421,PAULINA ALMENDRITA SOTELO VALERIANO,25493893.0,NaN,NaN,NaN,NaN,NaN,NaN,16 Feb 2026,01 Abr 2026,03 Ago 2026,4:00 pm,NaN,NaN
6,6.0,DEPARTAMENTO,704,LLANQUE FRAQUITA MANINO HELAMAN,40654929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Jul 2025,14 Oct 2025,22 Jul 2026,11:00 am,NaN,NaN


In [345]:
print(f'Total Entregas Pestaña 2: {df_procesos.shape[0]} clientes')
df_procesos.sample(2)


Total Entregas Pestaña 2: 66 clientes


,#,Tipo,N° Unidad,Comprador,% Avance,F. Separación,unidad_key,codigo_unidad
1,2,DEPARTAMENTO,402,BLAZ MARIÑO FRANK,0.1,06 Jun 2026,402,FX-402
44,45,DEPARTAMENTO,1204,POMALAZA INGA DANILO NILTON,0.2,31 Dic 2025,1204,FX-1204


### 2) Right

In [346]:
print(f'Total Propietarios: {df_propietarios.shape[0]} clientes')
df_propietarios.sample(2)

Total Propietarios: 1279 clientes


,documento,celulares,email,telefono,nombres,apellidos,celular_clean,celular_len,celular_ok,email_clean,email_ok
94718,07627686,+51999924892,carlos_anaka@yahoo.es,,Carlos Henry,Onaka Nuñez,999924892,9,True,carlos_anaka@yahoo.es,True
136994,40991505,+51999133778,anibalry18@gmail.com,,Anibal Erick,Recuay Uribe,999133778,9,True,anibalry18@gmail.com,True


### Dni para 2 propietarios diferente

In [347]:
df_propietarios["documento_key"] = (
    df_propietarios["documento"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(8)
)

df_propietarios.loc[
    df_propietarios["documento_key"] == "25493893",
    ["documento", "documento_key"]
] = ["45063421", "45063421"]

## 3) Merge

### PESTAÑA 1

In [348]:
# df_propietarios = documento
# df_entregas_completo = df_entregas.merge(df_propietarios, how='left', on='DNI')

In [349]:
import pandas as pd

# Copias para no tocar los dataframes originales
entregas = df_entregas.copy()
propietarios = df_propietarios.copy()

# 1) Normalizar llave DNI / documento
def limpiar_documento(serie):
    return (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)      # elimina .0 si vino de Excel
        .str.replace(r"\D", "", regex=True)        # deja solo números
        .replace("", pd.NA)
    )

entregas["dni_key"] = limpiar_documento(entregas["DNI"])
propietarios["documento_key"] = limpiar_documento(propietarios["documento"])

# 2) Opcional pero recomendado: quedarte solo con documentos válidos de 8 dígitos
entregas.loc[entregas["dni_key"].str.len() != 8, "dni_key"] = pd.NA
propietarios.loc[propietarios["documento_key"].str.len() != 8, "documento_key"] = pd.NA

# 3) Evitar duplicados en propietarios para no multiplicar filas
cols_propietarios = [
    "documento_key",
    "celulares",
    "celular_clean",
    "email",
    "email_clean",
    "telefono",
    "nombres"
]

propietarios_merge = (
    propietarios[cols_propietarios]
    .dropna(subset=["documento_key"])
    .drop_duplicates(subset=["documento_key"], keep="first")
)

# 4) Left merge: mantiene todas las filas de entregas
df_entregas_merge = entregas.merge(
    propietarios_merge,
    how="left",
    left_on="dni_key",
    right_on="documento_key",
    validate="m:1",
    indicator=True
)

df_entregas_merge.head()

,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.,dni_key,documento_key,celulares,celular_clean,email,email_clean,telefono,nombres,_merge
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN,45816592,45816592,+51999237144,999237144,peraltav.lisbeth@gmail.com,peraltav.lisbeth@gmail.com,,Lisbeth Sofia,both
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN,07990374,07990374,+51970465167,970465167,matoskina@gmail.com,matoskina@gmail.com,,monica,both
2,3.0,DEPARTAMENTO,1503,DEL VALLE VARILLAS JOSUE HAMILTON,71300365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Ago 2025,12 Sep 2025,20 Jul 2026,2:00 pm,NaN,NaN,71300365,71300365,+51962298910,962298910,josuedv13@hotmail.com,josuedv13@hotmail.com,-,josue HAMILTON,both
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both
4,NaN,ESTACIONAMIENTO,37,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Sep 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both


### Validaciones

In [350]:
df_entregas_merge["_merge"].value_counts()

_merge
both          39
left_only      0
right_only     0
Name: count, dtype: int64

In [351]:
df_entregas_merge[df_entregas_merge["_merge"] == "left_only"][
    ["DNI", "dni_key", "Comprador", "Tipo Unidad", "N° Unidad", "celular_clean"]
].head(20)

,DNI,dni_key,Comprador,Tipo Unidad,N° Unidad,celular_clean


In [352]:
df_entregas_merge

,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.,dni_key,documento_key,celulares,celular_clean,email,email_clean,telefono,nombres,_merge
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN,45816592,45816592,+51999237144,999237144,peraltav.lisbeth@gmail.com,peraltav.lisbeth@gmail.com,,Lisbeth Sofia,both
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN,07990374,07990374,+51970465167,970465167,matoskina@gmail.com,matoskina@gmail.com,,monica,both
2,3.0,DEPARTAMENTO,1503,DEL VALLE VARILLAS JOSUE HAMILTON,71300365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Ago 2025,12 Sep 2025,20 Jul 2026,2:00 pm,NaN,NaN,71300365,71300365,+51962298910,962298910,josuedv13@hotmail.com,josuedv13@hotmail.com,-,josue HAMILTON,both
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both
4,NaN,ESTACIONAMIENTO,37,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Sep 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both
5,5.0,DEPARTAMENTO,804,ROMO AGUIRRE LUIGUI ROLANDO,74827647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20 Dic 2024,02 Oct 2025,22 Jul 2026,9:00 am,NaN,NaN,74827647,74827647,+51932750569,932750569,romoluigui7@gmail.com,romoluigui7@gmail.com,-,Luigui ROLANDO,both
6,6.0,DEPARTAMENTO,704,LLANQUE FRAQUITA MANINO HELAMAN,40654929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Jul 2025,14 Oct 2025,22 Jul 2026,11:00 am,NaN,NaN,40654929,40654929,+51952852015,952852015,marinoii@hotmail.com,marinoii@hotmail.com,,Mariano Helaman,both
7,7.0,DEPARTAMENTO,1705,BULEJE CALDERON WILDER WILFREDO,71477258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,30 Oct 2025,22 Jul 2026,2:00 pm,NaN,NaN,71477258,71477258,+51986687399,986687399,wilderbuleje@hotmail.com,wilderbuleje@hotmail.com,,Wilder Wilfredo,both
8,8.0,DEPARTAMENTO,303,ALDERETE SOLIS SADITH PATRICIA,46137521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,31 Oct 2025,22 Jul 2026,4:00 pm,NaN,NaN,46137521,46137521,+51987469664,987469664,spatricia.2109@GMAIL.COM,spatricia.2109@gmail.com,987469664,SADITH PATRICIA,both
9,9.0,DEPARTAMENTO,802,BELLIDO PANTOJA ROSA MARIA,73097248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21 Ago 2025,24 Dic 2025,24 Jul 2026,9:00 am,NaN,NaN,73097248,73097248,+51965451936,965451936,rosabellidop23@gmail.com,rosabellidop23@gmail.com,,Rosa maria,both


In [353]:
df_separaciones

,nombre_proyecto,tipo_unidad_principal,codigo_unidad,documento_cliente,nombres_usuario
183,Fenix,departamento flat,FX-1105,41575485,Ana Lucia DelSolar
220,Fenix,departamento flat,FX-703,41543675,Ana Lucia DelSolar
242,Fenix,estacionamiento simple,FX-E35,29352434,Edwing Soller
243,Fenix,departamento flat,FX-1605,29352434,Edwing Soller
325,Fenix,departamento flat,FX-1001,44129606,Yomira Pacheco
...,...,...,...,...,...
4463,Fenix,departamento flat,FX-502,40485909,Luis Peñaloza
4480,Fenix,departamento flat,FX-1502,76292612,Joceline Arizabal Corimanya
4607,Fenix,departamento flat,FX-802,73097248,Joceline Arizabal Corimanya
4622,Fenix,estacionamiento con depósito,FX-E4,49034354,Ivan La Riva


In [354]:
import pandas as pd

# Copias para no tocar los dataframes originales
entregas = df_entregas_merge.copy()
propietarios = df_separaciones.copy()

# 1) Normalizar llave DNI / documento
def limpiar_documento(serie):
    return (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)      # elimina .0 si vino de Excel
        .str.replace(r"\D", "", regex=True)        # deja solo números
        .replace("", pd.NA)
    )

# Crear llaves limpias
entregas["dni_key"] = limpiar_documento(entregas["DNI"])
propietarios["documento_key"] = limpiar_documento(propietarios["documento_cliente"])

# 2) Quedarte solo con documentos válidos de 8 dígitos
entregas.loc[entregas["dni_key"].str.len() != 8, "dni_key"] = pd.NA
propietarios.loc[propietarios["documento_key"].str.len() != 8, "documento_key"] = pd.NA

# 3) Evitar duplicados en propietarios para no multiplicar filas
cols_propietarios = [
    "documento_key",
    "documento_cliente",
    "nombres_usuario", 
    "codigo_unidad"
]

propietarios_merge = (
    propietarios[cols_propietarios]
    .dropna(subset=["documento_key"])
    .drop_duplicates(subset=["documento_key"], keep="first")
)

# 4) Left merge: mantiene todas las filas de entregas
df_entregas_2_merge = entregas.merge(
    propietarios_merge,
    how="left",
    left_on="dni_key",
    right_on="documento_key",
    validate="m:1",
    indicator="merge_propietarios"
)

df_entregas_2_merge.head()

,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.,dni_key,documento_key_x,celulares,celular_clean,email,email_clean,telefono,nombres,_merge,documento_key_y,documento_cliente,nombres_usuario,codigo_unidad,merge_propietarios
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN,45816592,45816592,+51999237144,999237144,peraltav.lisbeth@gmail.com,peraltav.lisbeth@gmail.com,,Lisbeth Sofia,both,45816592,45816592,Edwing Soller,FX-1704,both
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN,07990374,07990374,+51970465167,970465167,matoskina@gmail.com,matoskina@gmail.com,,monica,both,07990374,07990374,Joceline Arizabal Corimanya,FX-904,both
2,3.0,DEPARTAMENTO,1503,DEL VALLE VARILLAS JOSUE HAMILTON,71300365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Ago 2025,12 Sep 2025,20 Jul 2026,2:00 pm,NaN,NaN,71300365,71300365,+51962298910,962298910,josuedv13@hotmail.com,josuedv13@hotmail.com,-,josue HAMILTON,both,71300365,71300365,Yomira Pacheco,FX-1503,both
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both,47160251,47160251,Ana Lucia DelSolar,FX-1702,both
4,NaN,ESTACIONAMIENTO,37,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Sep 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both,47160251,47160251,Ana Lucia DelSolar,FX-1702,both


## PESTAÑA 2

In [355]:
df_procesos.sample(6)

,#,Tipo,N° Unidad,Comprador,% Avance,F. Separación,unidad_key,codigo_unidad
46,47,DEPARTAMENTO,602,PRINCIPE COLLAZOS JUNIOR VALERIO,0.050,31 Dic 2025,602,FX-602
43,44,DEPARTAMENTO,202,PIÑA ORTIZ HELLEN PATRICIA,0.565,13 Nov 2025,202,FX-202
51,52,ESTACIONAMIENTO,9,ROMO AGUIRRE LUIGUI ROLANDO,0.708,07 Mar 2025,9,FX-E9
42,43,ESTACIONAMIENTO,5,PIÑA ORTIZ HELLEN PATRICIA,0.065,13 Nov 2025,5,FX-E5
2,3,DEPARTAMENTO,1201,CACERES VALVERDE HENRY ENRIQUE,0.003,16 Jun 2026,1201,FX-1201
49,50,ESTACIONAMIENTO,40,ROMERO PLASENCIA ALFREDO FRANCISCO,0.500,04 Ago 2025,40,FX-E40


In [356]:
df_separaciones

,nombre_proyecto,tipo_unidad_principal,codigo_unidad,documento_cliente,nombres_usuario
183,Fenix,departamento flat,FX-1105,41575485,Ana Lucia DelSolar
220,Fenix,departamento flat,FX-703,41543675,Ana Lucia DelSolar
242,Fenix,estacionamiento simple,FX-E35,29352434,Edwing Soller
243,Fenix,departamento flat,FX-1605,29352434,Edwing Soller
325,Fenix,departamento flat,FX-1001,44129606,Yomira Pacheco
...,...,...,...,...,...
4463,Fenix,departamento flat,FX-502,40485909,Luis Peñaloza
4480,Fenix,departamento flat,FX-1502,76292612,Joceline Arizabal Corimanya
4607,Fenix,departamento flat,FX-802,73097248,Joceline Arizabal Corimanya
4622,Fenix,estacionamiento con depósito,FX-E4,49034354,Ivan La Riva


In [357]:
df_propietarios

,documento,celulares,email,telefono,nombres,apellidos,celular_clean,celular_len,celular_ok,email_clean,email_ok,documento_key
70,46738455,+51948630997,WMRODRIGUEZS@OUTLOOK.COM,,WALTER MAXIMO,RODRIGUEZ SANDOVAL,948630997,9,True,wmrodriguezs@outlook.com,True,46738455
214,44612849,+51966769565,lizguerraz3008@gmail.com,-,ZAIDA LIZ,GUERRA ZERPA,966769565,9,True,lizguerraz3008@gmail.com,True,44612849
501,72488955,+51965707584,pmmr.2020@gmail.com,,Pedro Moises,Martinez Robles,965707584,9,True,pmmr.2020@gmail.com,True,72488955
551,44013049,+51979328785,johanst1009@gmail.com,,Silver Johanson,Sucso Tineo,979328785,9,True,johanst1009@gmail.com,True,44013049
560,45632301,+420773621533,DANIEL4289@OUTLOOK.COM,,LUIS DANIEL,GUTIERREZ ALMIDON,420773621533,12,False,daniel4289@outlook.com,True,45632301
...,...,...,...,...,...,...,...,...,...,...,...,...
136972,40524906,+51963936078,yhosmy@hotmail.com,,Hayme Joeicy,Rivera Mayta,963936078,9,True,yhosmy@hotmail.com,True,40524906
136990,07733980,+17869557376,ANAKINKOS3399@YAHOO.COM,,Ana Rosario,Zevallos Machuca de Rivero,17869557376,11,False,anakinkos3399@yahoo.com,True,07733980
136994,40991505,+51999133778,anibalry18@gmail.com,,Anibal Erick,Recuay Uribe,999133778,9,True,anibalry18@gmail.com,True,40991505
136995,43971806,+51998454444,KARLIZ_39@HOTMAIL.COM,,LIZ KARINA,QUISPE SANCHEZ,998454444,9,True,karliz_39@hotmail.com,True,43971806


In [358]:
import pandas as pd

# Copias para no tocar los dataframes originales
procesos     = df_procesos.copy()
procesos_separaciones = df_separaciones.copy()

# 1) Normalizar llave DNI / documento
def limpiar_documento(serie):
    return (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)      # elimina .0 si vino de Excel
        .str.replace(r"\D", "", regex=True)        # deja solo números
        .replace("", pd.NA)
    )

procesos["unidad_key"] = limpiar_documento(procesos["codigo_unidad"])
procesos_separaciones["unidad_key"] = limpiar_documento(procesos_separaciones["codigo_unidad"])

# 2) Opcional pero recomendado: quedarte solo con documentos válidos de 8 dígitos
#procesos.loc[procesos["unidad_key"].str.len() != 8, "unidad_key"] = pd.NA
#propietarios.loc[propietarios["unidad_key"].str.len() != 8, "unidad_key"] = pd.NA

# 3) Evitar duplicados en propietarios para no multiplicar filas
cols_propietarios = [
    "unidad_key",
    "nombres_usuario",
    "documento_cliente"#,
    # "email",
    # "email_clean",
    # "telefono",
    # "nombres"
]

propietarios_merge = (
    procesos_separaciones[cols_propietarios]
    .dropna(subset=["unidad_key"])
    .drop_duplicates(subset=["unidad_key"], keep="first")
)

# 4) Left merge: mantiene todas las filas de entregas
df_procesos_merge = procesos.merge(
    propietarios_merge,
    how="left",
    left_on="unidad_key",
    right_on="unidad_key",
    validate="m:1",
    indicator=True
)

df_procesos_merge.head()

,#,Tipo,N° Unidad,Comprador,% Avance,F. Separación,unidad_key,codigo_unidad,nombres_usuario,documento_cliente,_merge
0,1,ESTACIONAMIENTO,7,BELLIDO PANTOJA ROSA MARIA,0.770,24 Jun 2025,7,FX-E7,Joceline Arizabal Corimanya,73097248,both
1,2,DEPARTAMENTO,402,BLAZ MARIÑO FRANK,0.100,06 Jun 2026,402,FX-402,Nelly Román Villavicencio,25731745,both
2,3,DEPARTAMENTO,1201,CACERES VALVERDE HENRY ENRIQUE,0.003,16 Jun 2026,1201,FX-1201,Percy Soto,70459143,both
3,4,DEPARTAMENTO,903,CAMAHUALI HUAMAN NANCY LIZ,0.249,06 Abr 2025,903,FX-903,Sonia Paredes,auto-21119534187,both
4,5,ESTACIONAMIENTO,32,CAMAHUALI HUAMAN NANCY LIZ,0.000,NaN,32,FX-E32,NaN,NaN,left_only


In [359]:
import pandas as pd

# Copias para no tocar los dataframes originales
entregas = df_procesos_merge.copy()
propietarios = df_propietarios.copy()

# 1) Normalizar llave DNI / documento
def limpiar_documento(serie):
    return (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)      # elimina .0 si vino de Excel
        .str.replace(r"\D", "", regex=True)        # deja solo números
        .replace("", pd.NA)
    )

entregas["dni_key"] = limpiar_documento(entregas["documento_cliente"])
propietarios["documento_key"] = limpiar_documento(propietarios["documento"])

# 2) Opcional pero recomendado: quedarte solo con documentos válidos de 8 dígitos
entregas.loc[entregas["dni_key"].str.len() != 8, "dni_key"] = pd.NA
propietarios.loc[propietarios["documento_key"].str.len() != 8, "documento_key"] = pd.NA

# 3) Evitar duplicados en propietarios para no multiplicar filas
cols_propietarios = [
    "documento_key",
    "celulares",
    "celular_clean",
    "email",
    "email_clean",
    "telefono",
    "nombres"
]

propietarios_merge = (
    propietarios[cols_propietarios]
    .dropna(subset=["documento_key"])
    .drop_duplicates(subset=["documento_key"], keep="first")
)

# 4) Left merge: mantiene todas las filas de entregas
df_entregas_merge = entregas.merge(
    propietarios_merge,
    how="left",
    left_on="dni_key",
    right_on="documento_key",
    validate="m:1",
    #indicator=True
)

df_entregas_merge.to_excel('En Proceso.xlsx', index=False)
df_entregas_merge.head()

,#,Tipo,N° Unidad,Comprador,% Avance,F. Separación,unidad_key,codigo_unidad,nombres_usuario,documento_cliente,_merge,dni_key,documento_key,celulares,celular_clean,email,email_clean,telefono,nombres
0,1,ESTACIONAMIENTO,7,BELLIDO PANTOJA ROSA MARIA,0.770,24 Jun 2025,7,FX-E7,Joceline Arizabal Corimanya,73097248,both,73097248,73097248,+51965451936,965451936,rosabellidop23@gmail.com,rosabellidop23@gmail.com,,Rosa maria
1,2,DEPARTAMENTO,402,BLAZ MARIÑO FRANK,0.100,06 Jun 2026,402,FX-402,Nelly Román Villavicencio,25731745,both,25731745,25731745,+51976362387,976362387,FRANKBLAZ01@GMAIL.COM,frankblaz01@gmail.com,,FRANK
2,3,DEPARTAMENTO,1201,CACERES VALVERDE HENRY ENRIQUE,0.003,16 Jun 2026,1201,FX-1201,Percy Soto,70459143,both,70459143,70459143,+51995966779,995966779,henrycaceresv@gmail.com,henrycaceresv@gmail.com,,HENRY ENRIQUE
3,4,DEPARTAMENTO,903,CAMAHUALI HUAMAN NANCY LIZ,0.249,06 Abr 2025,903,FX-903,Sonia Paredes,auto-21119534187,both,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
4,5,ESTACIONAMIENTO,32,CAMAHUALI HUAMAN NANCY LIZ,0.000,NaN,32,FX-E32,NaN,NaN,left_only,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN


## Entrega Listo

In [360]:
print(f'Total Entregas Pestaña 1: {df_entregas_2_merge.shape[0]} clientes')
col_contacto = ['nombre_proyecto','tipo_unidad_principal', 'codigo_unidad' , 'documento_cliente', 'nombres_usuario']
df_separaciones = df_separaciones[col_contacto]
df_entregas_2_merge.to_excel('Entregas.xlsx', index=False)
df_entregas_2_merge.sample(15)

Total Entregas Pestaña 1: 39 clientes


,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.,dni_key,documento_key_x,celulares,celular_clean,email,email_clean,telefono,nombres,_merge,documento_key_y,documento_cliente,nombres_usuario,codigo_unidad,merge_propietarios
21,19.0,DEPARTAMENTO,503,FIERRO ROSALES JAVIER EDWIN,06010468,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Ene 2026,31 Mar 2026,03 Ago 2026,2:00 pm,NaN,NaN,06010468,06010468,+51993542688,993542688,jefr18nov@gmail.com,jefr18nov@gmail.com,,Edwin Javier,both,06010468,06010468,Ana Lucia DelSolar,FX-503,both
34,29.0,DEPARTAMENTO,1001,LOPEZ RAMON ANA MAGDALENA,45677959,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Mar 2026,27 May 2026,10 Ago 2026,9:00 am,NaN,NaN,45677959,45677959,+51969963071,969963071,magdalenalopez@gmail.com,magdalenalopez@gmail.com,,Ana Magdalena,both,<NA>,NaN,NaN,NaN,left_only
8,8.0,DEPARTAMENTO,303,ALDERETE SOLIS SADITH PATRICIA,46137521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,31 Oct 2025,22 Jul 2026,4:00 pm,NaN,NaN,46137521,46137521,+51987469664,987469664,spatricia.2109@GMAIL.COM,spatricia.2109@gmail.com,987469664,SADITH PATRICIA,both,46137521,46137521,Ivan La Riva,FX-303,both
22,20.0,DEPARTAMENTO,205,TORRES SOTELO MILAGROS MATILDE,45063421,PAULINA ALMENDRITA SOTELO VALERIANO,25493893.0,NaN,NaN,NaN,NaN,NaN,NaN,16 Feb 2026,01 Abr 2026,03 Ago 2026,4:00 pm,NaN,NaN,45063421,45063421,+51914591452,914591452,paulinasotelo@gmail.com,paulinasotelo@gmail.com,,Paulina Almendrita,both,<NA>,NaN,NaN,NaN,left_only
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN,45816592,45816592,+51999237144,999237144,peraltav.lisbeth@gmail.com,peraltav.lisbeth@gmail.com,,Lisbeth Sofia,both,45816592,45816592,Edwing Soller,FX-1704,both
31,27.0,DEPARTAMENTO,1203,MELGAR SALINAS ANTONY JOSUHE,71251490,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31 Mar 2026,05 May 2026,07 Ago 2026,2:00 pm,NaN,NaN,71251490,71251490,+51937339489,937339489,melgarantony@gmail.com,melgarantony@gmail.com,,Antony Josuhe,both,71251490,71251490,Ivan La Riva,FX-E43,both
20,18.0,DEPARTAMENTO,601,JULIAN GOMEZ ROSA YASIDH,70361676,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,09 Jul 2025,31 Mar 2026,03 Ago 2026,11:00 am,NaN,NaN,70361676,70361676,+51963893562,963893562,rosa.julian@unmsm.edu.pe,rosa.julian@unmsm.edu.pe,,Rosa Yasidh,both,70361676,70361676.,Luis Peñaloza,FX-601,both
5,5.0,DEPARTAMENTO,804,ROMO AGUIRRE LUIGUI ROLANDO,74827647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20 Dic 2024,02 Oct 2025,22 Jul 2026,9:00 am,NaN,NaN,74827647,74827647,+51932750569,932750569,romoluigui7@gmail.com,romoluigui7@gmail.com,-,Luigui ROLANDO,both,74827647,74827647,Yomira Pacheco,FX-E9,both
26,23.0,DEPARTAMENTO,1005,VILAVILA PALLQUI VALIA MARUCIA,72945691,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10 Oct 2025,25 Abr 2026,05 Ago 2026,2:00 pm,NaN,NaN,72945691,72945691,+51949332880,949332880,valiam016@gmail.com,valiam016@gmail.com,,Valia Marucia,both,72945691,72945691,Ana Lucia DelSolar,FX-1005,both
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN,07990374,07990374,+51970465167,970465167,matoskina@gmail.com,matoskina@gmail.com,,monica,both,07990374,07990374,Joceline Arizabal Corimanya,FX-904,both


In [361]:
print(f'Total Entregas Pestaña 1: {df_entregas_merge.shape[0]} clientes')

df_entregas_merge.sample(15)
df_entregas_2_merge

Total Entregas Pestaña 1: 66 clientes


,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.,dni_key,documento_key_x,celulares,celular_clean,email,email_clean,telefono,nombres,_merge,documento_key_y,documento_cliente,nombres_usuario,codigo_unidad,merge_propietarios
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN,45816592,45816592,+51999237144,999237144,peraltav.lisbeth@gmail.com,peraltav.lisbeth@gmail.com,,Lisbeth Sofia,both,45816592,45816592,Edwing Soller,FX-1704,both
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN,07990374,07990374,+51970465167,970465167,matoskina@gmail.com,matoskina@gmail.com,,monica,both,07990374,07990374,Joceline Arizabal Corimanya,FX-904,both
2,3.0,DEPARTAMENTO,1503,DEL VALLE VARILLAS JOSUE HAMILTON,71300365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Ago 2025,12 Sep 2025,20 Jul 2026,2:00 pm,NaN,NaN,71300365,71300365,+51962298910,962298910,josuedv13@hotmail.com,josuedv13@hotmail.com,-,josue HAMILTON,both,71300365,71300365,Yomira Pacheco,FX-1503,both
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both,47160251,47160251,Ana Lucia DelSolar,FX-1702,both
4,NaN,ESTACIONAMIENTO,37,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Sep 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both,47160251,47160251,Ana Lucia DelSolar,FX-1702,both
5,5.0,DEPARTAMENTO,804,ROMO AGUIRRE LUIGUI ROLANDO,74827647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20 Dic 2024,02 Oct 2025,22 Jul 2026,9:00 am,NaN,NaN,74827647,74827647,+51932750569,932750569,romoluigui7@gmail.com,romoluigui7@gmail.com,-,Luigui ROLANDO,both,74827647,74827647,Yomira Pacheco,FX-E9,both
6,6.0,DEPARTAMENTO,704,LLANQUE FRAQUITA MANINO HELAMAN,40654929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Jul 2025,14 Oct 2025,22 Jul 2026,11:00 am,NaN,NaN,40654929,40654929,+51952852015,952852015,marinoii@hotmail.com,marinoii@hotmail.com,,Mariano Helaman,both,40654929,40654929,Antonio Martinez,FX-704,both
7,7.0,DEPARTAMENTO,1705,BULEJE CALDERON WILDER WILFREDO,71477258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,30 Oct 2025,22 Jul 2026,2:00 pm,NaN,NaN,71477258,71477258,+51986687399,986687399,wilderbuleje@hotmail.com,wilderbuleje@hotmail.com,,Wilder Wilfredo,both,71477258,71477258,Antonio Martinez,FX-1705,both
8,8.0,DEPARTAMENTO,303,ALDERETE SOLIS SADITH PATRICIA,46137521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,31 Oct 2025,22 Jul 2026,4:00 pm,NaN,NaN,46137521,46137521,+51987469664,987469664,spatricia.2109@GMAIL.COM,spatricia.2109@gmail.com,987469664,SADITH PATRICIA,both,46137521,46137521,Ivan La Riva,FX-303,both
9,9.0,DEPARTAMENTO,802,BELLIDO PANTOJA ROSA MARIA,73097248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21 Ago 2025,24 Dic 2025,24 Jul 2026,9:00 am,NaN,NaN,73097248,73097248,+51965451936,965451936,rosabellidop23@gmail.com,rosabellidop23@gmail.com,,Rosa maria,both,73097248,73097248,Joceline Arizabal Corimanya,FX-E7,both
